# Consultas SQL: Clima y Rendimiento de Cultivos

Con el dataset final ya validado en el EDA, este notebook lo pasa a SQL: se carga a una base SQLite y se diseñan las views que van a alimentar el dashboard en Power BI. La idea es que cada view encapsule una de las preguntas que ya respondimos con pandas en el notebook `04` — y que podamos verificarlo comparando los resultados número contra número.

## 1. Librerías, Base de Datos y Carga de Datos

### 1.1 Librerías

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

### 1.2 Creación de la Base de Datos SQLite y Carga del Dataset

In [2]:
RUTA_PROYECTO = Path().resolve()
if RUTA_PROYECTO.name == "notebook":  # los notebooks se ejecutan desde la carpeta notebook/
    RUTA_PROYECTO = RUTA_PROYECTO.parent
RUTA_PROCESSED = RUTA_PROYECTO / "data" / "processed"
RUTA_SQL = RUTA_PROYECTO / "sql"
RUTA_SQL.mkdir(parents=True, exist_ok=True)

conexion = sqlite3.connect(RUTA_SQL / "proyecto_agro.db")

df_final = pd.read_csv(RUTA_PROCESSED / "fusion_clima_cultivos_final.csv")
df_final.to_sql('fusion_clima_cultivos', conexion, if_exists='replace', index=False)

print("Tabla cargada. Filas:", df_final.shape[0])

Tabla cargada. Filas: 138926


## 2. Creación de Views

### 2.1 Vista: Rendimiento Nacional por Cultivo y Año

In [3]:
cursor = conexion.cursor()

cursor.execute("DROP VIEW IF EXISTS v_rendimiento_nacional_cultivo_anio")

cursor.execute("""
CREATE VIEW v_rendimiento_nacional_cultivo_anio AS
SELECT
    cultivo,
    anio,
    SUM(produccion_tm) AS produccion_total_tm,
    SUM(superficie_cosechada_ha) AS superficie_cosechada_total_ha,
    (SUM(produccion_tm) * 1000.0) / SUM(superficie_cosechada_ha) AS rendimiento_ponderado_kgxha
FROM fusion_clima_cultivos
GROUP BY cultivo, anio
""")

conexion.commit()

In [4]:
# Verificación: coincide con los valores de avena 1923-1927 obtenidos en pandas (notebook 04)
pd.read_sql("""
    SELECT * FROM v_rendimiento_nacional_cultivo_anio
    WHERE cultivo = 'avena' AND anio BETWEEN 1923 AND 1927
""", conexion)

,cultivo,anio,produccion_total_tm,superficie_cosechada_total_ha,rendimiento_ponderado_kgxha
0,avena,1923,1104052.0,859596.0,1284.384757
1,avena,1924,775611.0,690902.0,1122.606390
2,avena,1925,1166682.0,905407.0,1288.571880
3,avena,1926,961998.0,898068.0,1071.186146
4,avena,1927,759000.0,705430.0,1075.939498


**Nota — `v_rendimiento_nacional_cultivo_anio`**

Vista que replica el cálculo de rendimiento ponderado nacional por cultivo y año hecho en pandas (notebook `04`, Sección 4.2). Verificada contra los valores de avena 1923-1927, con coincidencia exacta. Será la fuente para el gráfico de evolución temporal por cultivo en Power BI.

### 2.2 Vista: Rendimiento por Provincia y Cultivo (Histórico Completo)

In [5]:
cursor.execute("DROP VIEW IF EXISTS v_rendimiento_provincia_cultivo")

cursor.execute("""
CREATE VIEW v_rendimiento_provincia_cultivo AS
SELECT
    provincia,
    cultivo,
    SUM(produccion_tm) AS produccion_total_tm,
    SUM(superficie_cosechada_ha) AS superficie_cosechada_total_ha,
    (SUM(produccion_tm) * 1000.0) / SUM(superficie_cosechada_ha) AS rendimiento_ponderado_kgxha
FROM fusion_clima_cultivos
WHERE provincia IS NOT NULL
GROUP BY provincia, cultivo
""")

conexion.commit()

In [6]:
# Verificación: coincide con la fila de Buenos Aires del heatmap del notebook 04 (Sección 3)
pd.read_sql("""
    SELECT * FROM v_rendimiento_provincia_cultivo
    WHERE provincia = 'BUENOS AIRES'
""", conexion)

,provincia,cultivo,produccion_total_tm,superficie_cosechada_total_ha,rendimiento_ponderado_kgxha
0,BUENOS AIRES,avena,51562704.0,3.834199e+07,1344.810254
1,BUENOS AIRES,centeno,11465361.0,1.339955e+07,855.652818
2,BUENOS AIRES,girasol,100590681.0,6.158523e+07,1633.357315
3,BUENOS AIRES,maní,1069917.0,3.171340e+05,3373.706383
4,BUENOS AIRES,maíz,480221387.0,1.149463e+08,4177.788721
5,BUENOS AIRES,soja,377524919.0,1.431509e+08,2637.252282
6,BUENOS AIRES,sorgo,32164219.0,8.722770e+06,3687.385888
7,BUENOS AIRES,trigo,524405463.0,2.737666e+08,1915.520064


**Nota — `v_rendimiento_provincia_cultivo`**

Vista que replica el cálculo de rendimiento ponderado por provincia y cultivo hecho en pandas (notebook `04`, Sección 3). Incluye `WHERE provincia IS NOT NULL` para replicar el comportamiento de `groupby()` en pandas, que descarta por defecto las claves de agrupación nulas (a diferencia de SQL, donde `NULL` es un grupo válido). Verificada contra los valores de Buenos Aires para los 8 cultivos, con coincidencia exacta frente al notebook `04`. Es la vista histórica completa (1923-2025); será la fuente para el heatmap provincia×cultivo de "foto histórica" en Power BI.

### 2.3 Vista: Rendimiento por Provincia y Cultivo (Desde 2010)

In [7]:
cursor.execute("DROP VIEW IF EXISTS v_rendimiento_provincia_cultivo_reciente")
cursor.execute("""
CREATE VIEW v_rendimiento_provincia_cultivo_reciente AS
SELECT provincia, cultivo,
    SUM(produccion_tm) AS produccion_total_tm,
    SUM(superficie_cosechada_ha) AS superficie_cosechada_total_ha,
    (SUM(produccion_tm) * 1000.0) / SUM(superficie_cosechada_ha) AS rendimiento_ponderado_kgxha
FROM fusion_clima_cultivos
WHERE provincia IS NOT NULL AND anio >= 2010
GROUP BY provincia, cultivo
""")
conexion.commit()

In [8]:
# Verificación: Córdoba supera a Santiago del Estero en maíz dentro de la ventana reciente,
# igual que en la tabla pivot reciente del notebook 04 (Sección 3.3)
pd.read_sql("""
    SELECT provincia, cultivo, rendimiento_ponderado_kgxha
    FROM v_rendimiento_provincia_cultivo_reciente
    WHERE cultivo = 'maíz' AND provincia IN ('CÓRDOBA', 'SANTIAGO DEL ESTERO')
""", conexion)

,provincia,cultivo,rendimiento_ponderado_kgxha
0,CÓRDOBA,maíz,7237.527866
1,SANTIAGO DEL ESTERO,maíz,6246.291924


**Nota — `v_rendimiento_provincia_cultivo_reciente`**

Vista que replica el recorte temporal reciente (2010 en adelante) del notebook `04`, Sección 3.3, usada para corregir el sesgo de concentración de producción detectado al comparar Córdoba y Santiago del Estero en maíz. Misma lógica que `v_rendimiento_provincia_cultivo` (incluyendo `WHERE provincia IS NOT NULL`), con el filtro adicional `anio >= 2010`. Verificada contra la tabla pivot reciente del notebook `04`, con coincidencia exacta en los valores de Córdoba y Santiago del Estero (maíz). Será la fuente del segundo heatmap ("foto actual") en Power BI.

### 2.4 Vista: Rendimiento vs Variables Climáticas (1991 - actualidad)

In [9]:
cursor.execute("DROP VIEW IF EXISTS v_clima_rendimiento")
cursor.execute("""
CREATE VIEW v_clima_rendimiento AS
SELECT cultivo, provincia, anio,
    rendimiento_kgxha,
    TMAX, TMIN, TMEDIA, HELIOF, PRE_EST, PRECIP
FROM fusion_clima_cultivos
WHERE anio >= 1991
""")
conexion.commit()

In [10]:
# Verificación: 47.358 filas, igual que en el notebook 04, Sección 4.1
pd.read_sql("SELECT COUNT(*) AS filas FROM v_clima_rendimiento", conexion)

,filas
0,47358


**Nota — `v_clima_rendimiento`**

Vista a nivel de fila (sin agregación) con cultivo, provincia, año, rendimiento y las 6 variables climáticas usadas en el análisis de correlación del notebook `04` (`TMAX`, `TMIN`, `TMEDIA`, `HELIOF`, `PRE_EST`, `PRECIP`). Filtrada a `anio >= 1991` porque la cobertura climática del SMN recién comienza ese año. Se excluyen deliberadamente `VTO_MVEL`, `VEL_MED`, `HUM_REL`, `NUB_TOTAL` por estar nulas antes de 2021 y no haberse usado en ninguna conclusión del EDA. Verificada en cantidad de filas (47.358) contra el notebook `04`, Sección 4.1. Nota de diseño: al estar el clima asignado a nivel provincia+año, distintos departamentos comparten el mismo valor climático dentro del mismo año — el mismo artefacto de "bandas verticales" ya documentado en los scatter plots del EDA. Será la base para los scatter/correlación interactivos en Power BI.

## 3. Conclusiones del Notebook

Se construyó una base de datos SQLite (`proyecto_agro.db`) a partir del dataset final de fusión clima-cultivos, y se diseñaron 4 views que replican y encapsulan los análisis clave del EDA (notebook `04`):

- **`v_rendimiento_nacional_cultivo_anio`**: evolución del rendimiento ponderado nacional por cultivo y año.
- **`v_rendimiento_provincia_cultivo`**: rendimiento ponderado histórico (1923-2025) por provincia y cultivo.
- **`v_rendimiento_provincia_cultivo_reciente`**: la misma vista, acotada a 2010 en adelante, para corregir el sesgo de concentración de producción detectado entre Córdoba y Santiago del Estero.
- **`v_clima_rendimiento`**: dataset a nivel de fila (clima + rendimiento), acotado a 1991 en adelante, pensado como base para visualizaciones interactivas de correlación en Power BI.

Cada view fue verificada numéricamente contra los resultados ya validados en pandas (notebook `04`), con coincidencia exacta en todos los casos. Las views quedan exportadas a CSV como fuente de datos para la siguiente fase del proyecto: la construcción del dashboard en Power BI.

### 3.1 Exportación de las Views a CSV

In [11]:
RUTA_EXPORT = RUTA_PROYECTO / "powerbi"
RUTA_EXPORT.mkdir(parents=True, exist_ok=True)

vistas = [
    "v_rendimiento_nacional_cultivo_anio",
    "v_rendimiento_provincia_cultivo",
    "v_rendimiento_provincia_cultivo_reciente",
    "v_clima_rendimiento"
]

for vista in vistas:
    df_vista = pd.read_sql(f"SELECT * FROM {vista}", conexion)
    df_vista.to_csv(RUTA_EXPORT / f"{vista}.csv", index=False, encoding='utf-8')
    print(f"{vista}: {df_vista.shape}")

v_rendimiento_nacional_cultivo_anio: (701, 5)
v_rendimiento_provincia_cultivo: (160, 5)
v_rendimiento_provincia_cultivo_reciente: (105, 5)


v_clima_rendimiento: (47358, 10)
